In [159]:
!pip install pandas numpy transformers scikit-learn

import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split


## Download Real SMS Spam Dataset from Kaggle

To use Kaggle API:
1. Create a Kaggle account at https://www.kaggle.com
2. Go to Account Settings → API → Create New Token
3. This downloads `kaggle.json` - place it in `~/.kaggle/` directory
4. Run: `chmod 600 ~/.kaggle/kaggle.json`

In [160]:
# Install Kaggle API
!pip install kaggle

In [161]:
# Download UCI SMS Spam Collection Dataset from Kaggle
# Make sure you have set up your Kaggle API credentials first!
!kaggle datasets download -d uciml/sms-spam-collection-dataset -p ./data --unzip

Dataset URL: https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset
License(s): unknown
100%|█████████████████████████████████████████| 211k/211k [00:00<00:00, 241kB/s]



In [162]:
# Load the real SMS Spam dataset
sms_df_real = pd.read_csv('./data/spam.csv', encoding='latin-1')

# The dataset has extra unnamed columns, keep only the first two
sms_df_real = sms_df_real[['v1', 'v2']]
sms_df_real.columns = ['label', 'text']

print(f"Dataset shape: {sms_df_real.shape}")
print(f"\nLabel distribution:")
print(sms_df_real['label'].value_counts())
sms_df_real.head()

Dataset shape: (5572, 2)

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## Now use the real dataset (sms_df_real) instead of synthetic data

This UCI SMS Spam Collection contains 5,574 real SMS messages with realistic patterns. You should see more realistic accuracy scores (typically 95-98% instead of 100%).

In [163]:
# Create numeric labels for the real dataset
sms_df_real['label_num'] = sms_df_real['label'].map({'ham': 0, 'spam': 1})
print("Real dataset with label_num column:")
sms_df_real.head()

Real dataset with label_num column:


,label,text,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [164]:
sms_df=pd.read_csv('GL Notes/sms_spam_synthetic.csv', encoding='latin-1')
sms_df.shape
sms_df.head()   
sms_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    5000 non-null   str  
 1   label   5000 non-null   str  
dtypes: str(2)
memory usage: 78.3 KB


In [165]:
sms_df.sample(n=15)

,text,label
413,See you soon 16 please,ham
3093,I'll be late by 10 minutes yup,ham
4216,WIN CONGRATS BONUS DEAL HTTP://OFFER. GIFT IMM...,spam
89,Reached home,ham
4153,reply GIFT IMMEDIATELY http://offer. DEAL 100%...,spam
4853,PRIZE PRIZE 100% LOTTERY STOP PRIZE GUARANTEED...,spam
1099,Call me when you are free 7,ham
3700,Call me when you are free 20 home,ham
3202,Can you send the file? 21,ham
483,URGENT CONGRATS PRIZE BONUS GIFT HOT CASH $$$ ...,spam


In [166]:
sms_df.loc[0, "text"]

'Please check email yup shop'

In [167]:
sms_df.groupby("label").size()
print("Number of ham messages:", sms_df[sms_df["label"]=="ham"].shape[0])
print("Number of spam messages:", sms_df[sms_df["label"]=="spam"].shape[0])

print("probablity of Ham messages: <-- should be close to 0.82 which is the base level accuracy if we predict all messages as ham   ")
print(4100/(4100+900))
print("probablity of Spam messages:")
print(900/(4100+900))


Number of ham messages: 4100
Number of spam messages: 900
probablity of Ham messages: <-- should be close to 0.82 which is the base level accuracy if we predict all messages as ham   
0.82
probablity of Spam messages:
0.18


In [168]:
msg_num = np.random.randint(0, len(sms_df))
print("Randomly selected message:")
print(sms_df.loc[msg_num, "text"])
print("Label:", sms_df.loc[msg_num, "label"])

sms_df.isnull().sum()

Randomly selected message:
On my way 6 ok office
Label: ham


text     0
label    0
dtype: int64

In [169]:
sms_df['label_num'] = sms_df['label'].map({'ham': 0, 'spam': 1})
sms_df.head()
sms_df.sample(n=15)

,text,label,label_num
1807,Are we meeting today? 17,ham,0
2221,Good night 21 thanks shop,ham,0
1719,Meeting moved to tomorrow,ham,0
4359,Let's catch up later,ham,0
1795,Thanks for your help 13,ham,0
1957,Are we meeting today?,ham,0
1249,See you soon 24,ham,0
416,See you soon 25 sure,ham,0
3572,CLICK $$$ ACT IMMEDIATELY STOP VOUCHER LIMITED...,spam,1
1333,Let's catch up later thanks home,ham,0


# Create Training & Test DataSet

In [170]:
sms_train, sms_test, y_train, y_test = train_test_split(sms_df_real['text'], sms_df_real['label_num'], random_state=42)
sms_train.reset_index(inplace=  True, drop=True)
sms_test.reset_index(inplace=  True, drop=True)
y_train.reset_index(inplace=  True, drop=True)
y_test.reset_index(inplace=  True, drop=True)

print(sms_test.shape)
print(y_test.shape)

(1393,)
(1393,)


# Tokenization & Vectorization

In [171]:
from sklearn.feature_extraction.text import CountVectorizer
cvect = CountVectorizer(stop_words='english')
sms_train.head()

0                                    U can call now...
1    Tell them u have a headache and just want to u...
2    Never try alone to take the weight of a tear t...
3    Raji..pls do me a favour. Pls convey my Birthd...
4              What time. IÛ÷m out until prob 3 or so
Name: text, dtype: str

In [172]:
cvect.fit(sms_train)

cvect.vocabulary_

{'tell': 6265,
 'headache': 3157,
 'just': 3619,
 'want': 6834,
 'use': 6678,
 'hour': 3289,
 'sick': 5698,
 'time': 6393,
 'try': 6539,
 'weight': 6898,
 'tear': 6250,
 'comes': 1800,
 'ur': 6663,
 'heart': 3166,
 'falls': 2602,
 'eyes': 2577,
 'remember': 5270,
 'stupid': 6064,
 'friend': 2828,
 'share': 5623,
 'bslvyl': 1436,
 'raji': 5145,
 'pls': 4868,
 'favour': 2634,
 'convey': 1883,
 'birthday': 1275,
 'wishes': 6970,
 'nimya': 4447,
 'today': 6427,
 'prob': 5028,
 'hi': 3206,
 'kate': 3644,
 'lovely': 3939,
 'tonight': 6453,
 'ill': 3380,
 'phone': 4799,
 'tomorrow': 6444,
 'got': 3004,
 'sing': 5722,
 'guy': 3079,
 'gave': 2901,
 'card': 1544,
 'xxx': 7082,
 'usual': 6688,
 'iam': 3342,
 'fine': 2689,
 'happy': 3129,
 'amp': 892,
 'doing': 2259,
 'nope': 4474,
 'watching': 6855,
 'tv': 6561,
 'home': 3248,
 'going': 2976,
 'bored': 1347,
 'ps': 5074,
 'grown': 3057,
 'right': 5364,
 'wat': 6850,
 'tht': 6375,
 'incident': 3405,
 'awake': 1090,
 'snow': 5818,
 'good': 2986,
 '

In [173]:
len(cvect.vocabulary_)

7180

In [174]:
#Build Document Term Matrix DTM 
#  
x_train = cvect.transform(sms_train)
x_train.shape

(4179, 7180)

In [175]:
sms_train[0]

'U can call now...'

In [176]:
x_train[0]

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 0 stored elements and shape (1, 7180)>

In [177]:
print(x_train[0:1])

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 0 stored elements and shape (1, 7180)>


In [178]:
x_test = cvect.transform(sms_test)
x_test.shape

(1393, 7180)

In [179]:
# Building an SMS Classifier

from sklearn.neighbors import KNeighborsClassifier
kknn = KNeighborsClassifier(n_neighbors=5)
kknn.fit(x_train, y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [180]:
#Evaluation on Test Dataset 

from sklearn import metrics
metrics.accuracy_score(y_test, kknn.predict(x_test))

0.9117013639626705

In [181]:
metrics.accuracy_score(y_train, kknn.predict(x_train))

0.9246231155778895

In [182]:
#classifier using SVM

from sklearn.svm import SVC
svm = SVC(kernel='linear')
svm.fit(x_train, y_train)
metrics.accuracy_score(y_test, svm.predict(x_test))
metrics.accuracy_score(y_train, svm.predict(x_train))


1.0

In [183]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier()
rf_model.fit(x_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [184]:
metrics.accuracy_score(y_test, rf_model.predict(x_test))

0.9755922469490309

In [185]:
from sklearn.feature_extraction.text import TfidfVectorizer
tvect = TfidfVectorizer()
tvect.fit(sms_train)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1

In [186]:
len(tvect.vocabulary_)

7441

In [187]:
x_train_tfidf = tvect.transform(sms_train)
x_train_tfidf.shape


(4179, 7441)

In [188]:
print(x_train_tfidf[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3 stored elements and shape (1, 7441)>
  Coords	Values
  (0, 1552)	0.5542520595511462
  (0, 1576)	0.6054718381449722
  (0, 4648)	0.5711466604092693


In [189]:
x_test_tfidf = tvect.transform(sms_test)
x_test_tfidf.shape  

(1393, 7441)

In [190]:
# now we apply with Random Forest Classifier
rf_model_tfidf = RandomForestClassifier()
rf_model_tfidf.fit(x_train_tfidf, y_train)
metrics.accuracy_score(y_test, rf_model_tfidf.predict(x_test_tfidf))



0.9741564967695621